## **Tentatives d'installation de MMPose avec GPU**
Une de nos premières intentions était d'utiliser le modèle RTMPose de MMPose, pour étudier le mouvement d'une personne dans une vidéo. Or cette librairie utilise obligatoirement le GPU.

Nous avons tenté dans un premier temps d'installer MMPose à l'aide du GPU Nvidia T4 allouable par Google Colab.

---

Cette piste nous a mené de nombreux soucis comme la gestion et le patch des différentes versions de MMCV, MMDetection et MMEngine pour que tout fonctionne, ou alors tout simplement le fait que les GPU Nvidia T4 de Google n'était pas tout le temps à notre disposition.

Nous avons donc naturellement voulu faire tourner l'installation en local, mais des librairies comme PyTorch nécessitait des pilotes graphiques Nvidia (donc un GPU Nvidia) que nous n'avions pas dans nos machines respectives.

C'est pour cela que nous avons abandonné la piste RTMPose pour notre projet et que nous nous sommes concentrés sur le modèle HRNet. Voici le détail de nos essais :

---

On installe MMPose avec GPU puis on applique un patch permettant de corriger les problèmes de versions.

*IMPORTANT :* Après avoir exécuté cette cellule, il faut redémarrer la session (Exécution > Redémarrer la session).

In [ ]:
# 1. Nettoyage de l'environnement
%pip uninstall -y mmpose mmcv mmcv-lite mmdet mmengine

# 2. Installation de PyTorch 2.4 (Stable pour Python 3.12)
%pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

# Inutile dnas le cadre de notre projet et produit une erreur si on le laisse
%pip uninstall -y torchaudio

# 3. Installation de MMEngine depuis GITHUB (Source)
%pip install "git+https://github.com/open-mmlab/mmengine.git"

# 4. Installation de MMCV (Via WHEEL)
%pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4.0/index.html

# 5. Installation de MMDetection depuis GITHUB (Source)
%pip install "git+https://github.com/open-mmlab/mmdetection.git"

# 6. Installation de MMPose depuis GITHUB (Source)
!git clone https://github.com/open-mmlab/mmpose.git
%cd mmpose
%pip install -r requirements.txt
%pip install -e .
# Retour au dossier racine
%cd ..

## PATCH ##
print("\n" + "#" * 9 + "\n# PATCH #\n" + "#" * 9)

import os
import re

# Chemin du fichier
target_file = '/usr/local/lib/python3.12/dist-packages/mmdet/__init__.py'

# 1. Lecture
with open(target_file, 'r') as f:
    content = f.read()

# 2. Modification intelligente (On change la version MAX autorisée)
# On remplace '2.2.0' (ou autre) par '3.0.0' dans la définition de la variable
new_content = re.sub(
    r"mmcv_maximum_version = ['\"].*?['\"]",
    "mmcv_maximum_version = '3.0.0'",
    content
)

# 3. Écriture
with open(target_file, 'w') as f:
    f.write(new_content)

print("Patch appliqué : Redémarrez la session (Exécution > Redémarrer la session)")

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 30.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 109.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 62.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 125.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 782.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/1

*Veérification de l'installation :*

In [ ]:
import mmengine
import mmcv
import mmdet
import mmpose

print(f"MMEngine: {mmengine.__version__}")
print(f"MMCV:     {mmcv.__version__}")
print(f"MMDet:    {mmdet.__version__}")

MMEngine: 0.11.0rc2
MMCV:     2.2.0
MMDet:    3.3.0


In [ ]:
%pip show mmpose

Name: mmpose
Version: 1.3.2
Summary: OpenMMLab Pose Estimation Toolbox and Benchmark.
Home-page: https://github.com/open-mmlab/mmpose
Author: MMPose Contributors
Author-email: openmmlab@gmail.com
License: Apache License 2.0
Location: /content/mmpose
Editable project location: /content/mmpose
Requires: chumpy, json_tricks, matplotlib, munkres, numpy, opencv-python, pillow, scipy, torchvision, xtcocotools
Required-by: 


*Affichage de l'arborescence des fichiers pour visualiser où se trouvent les librairies :*

In [ ]:
!apt-get install -q tree
!tree

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 1s (53.9 kB/s)
Selecting previously unselected package tree.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
.
├── mmpose
│   ├── CITATION.cff
│   ├── configs
│   │   ├── animal_2d_keypoint
│   │   │   ├── README.md
│   │   │   ├── rtmpose
│   │   │   │   ├── ap10k
│   │   │   │   │   ├── rtmpose_ap10k.md
│   │   │   │   │   ├── rtmpose_ap10k.yml
│   │   │   │   │   └── rtmpose-m_8xb64-210e_ap10k-256x256.py


In [ ]:
import os
import torch

# 1. Suppression des anciens fichiers corrompus
if os.path.exists('checkpoints'):
    print("Suppression des anciens fichiers...")
    !rm -rf checkpoints
os.makedirs('checkpoints', exist_ok=True)

# 2. Définition des URLs officielles
# RTMDet-m (Détecteur)
url_det = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_m_8xb32-300e_coco/rtmdet_m_8xb32-300e_coco_20220719_112220-229f527c.pth'
# RTMPose-m (Squelette)
url_pose = 'https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-aic-coco_pt-aic-coco_420e-256x192-63eb25f7_20230126.pth'

# 3. Téléchargement via PyTorch (plus fiable que wget pour les redirections)
print("Téléchargement du détecteur (RTMDet)...")
try:
    torch.hub.download_url_to_file(url_det, 'checkpoints/rtmdet_m.pth')
except Exception as e:
    print(f"Erreur téléchargement détecteur : {e}")

print("\nTéléchargement du squelette (RTMPose)...")
try:
    torch.hub.download_url_to_file(url_pose, 'checkpoints/rtmpose_m.pth')
except Exception as e:
    print(f"Erreur téléchargement squelette : {e}")

# 4. Vérification de la réussite
print("\n--- Diagnostic des fichiers ---")
try:
    size_det = os.path.getsize('checkpoints/rtmdet_m.pth') / (1024 * 1024) # En MB
    size_pose = os.path.getsize('checkpoints/rtmpose_m.pth') / (1024 * 1024) # En MB

    print(f"RTMDet-m : {size_det:.2f} MB")
    print(f"RTMPose-m: {size_pose:.2f} MB")

    if size_det < 10 or size_pose < 10:
        print("\n❌ ALERTE : Les fichiers sont trop petits (< 10MB). Le lien est probablement bloqué.")
    else:
        print("\n✅ SUCCÈS : Les fichiers ont une taille cohérente. Tu peux relancer l'inférence !")
except FileNotFoundError:
    print("\n❌ ALERTE : Les fichiers n'ont pas été créés.")

Téléchargement du détecteur (RTMDet)...


100%|██████████| 214M/214M [00:01<00:00, 113MB/s]



Téléchargement du squelette (RTMPose)...


100%|██████████| 52.0M/52.0M [00:01<00:00, 36.7MB/s]


--- Diagnostic des fichiers ---
RTMDet-m : 213.91 MB
RTMPose-m: 52.01 MB

✅ SUCCÈS : Les fichiers ont une taille cohérente. Tu peux relancer l'inférence !


In [ ]:
import torch
from mmpose.apis import MMPoseInferencer

# --- 1. Définition des chemins LOCAUX (plus d'URL ici) ---
# Configs (déjà présentes grâce au git clone)
pose_config = 'mmpose/configs/body_2d_keypoint/rtmpose/coco/rtmpose-m_8xb256-420e_coco-256x192.py'
det_config  = 'mmpose/projects/rtmpose/rtmdet/person/rtmdet_m_640-8xb32_coco-person.py'

# Poids (téléchargés à l'étape précédente)
pose_weight_local = 'checkpoints/rtmpose_m.pth'
det_weight_local  = 'checkpoints/rtmdet_m.pth'

# --- 2. Initialisation ---
print("Chargement de l'inférenceur...")
inferencer = MMPoseInferencer(
    # Squelette
    pose2d=pose_config,
    pose2d_weights=pose_weight_local,

    # Détecteur
    det_model=det_config,
    det_weights=det_weight_local,

    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# --- 3. Traitement de la vidéo ---
video_path = 'input_video.mp4' # Vérifie que ce fichier est bien dans tes fichiers Colab
output_dir = 'output_results'

print(f"Traitement de la vidéo : {video_path}")

# On lance le générateur
result_generator = inferencer(
    video_path,
    out_dir=output_dir,
    vis_out_dir=output_dir,
    radius=4,
    thickness=2
)

# Boucle d'exécution
for result in result_generator:
    pass

print(f"Terminé ! Résultat sauvegardé dans {output_dir}/visualization/")

ModuleNotFoundError: No module named 'mmpose.apis'

---

*Rendu du groupe composé de :*
```
BABU Bastien
CLAUX Jérémy
PARODI Pierre
QASSEMYAR David
```